# Embedding, vector index and search (KAN-97)
Gözde Nur Özdemir · ELIZA · Sprint 2

**Goal:** turn the 105 chunks from `src/chunking.ipynb` into vectors, store them in a vector database, and return the 5 most relevant chunks for a question. This is the course's Sprint 2 baseline: *"the most basic vector database and retrieval over your documents"*, and the halfway check on 11/11: *"retrieval returns passages for a handful of test questions"*.

**Choices (from the design document):**
| Part | Choice | Decision |
|---|---|---|
| Embedding model | `BAAI/bge-base-en-v1.5`, 768 numbers per text | D10 |
| Vector store | Chroma, saved on disk, cosine distance | D11 |
| Keyword search | BM25 (`rank_bm25`) | D12 |
| Hybrid search | dense + BM25 merged with Reciprocal Rank Fusion, k = 60 | D12 |
| Top-k | 5 for now; chosen later with Recall@k on the `val` questions (3, 5 or 8) | D12 |

**Steps**
1. Load the settings and the chunks
2. Prepare the text: what the model reads, and stopwords for keyword search
3. Find the common words and phrases (candidates only, nothing is removed automatically)
4. Embed the chunks
5. Store them in Chroma
6. Build the keyword index (BM25)
7. Three ways to search: dense, BM25, hybrid
8. Try a handful of test questions
9. Compare the three ways on one question
10. What comes next

**How to run:** open this notebook from the `src/` folder. The first run downloads the model (about 440 MB). It runs on a normal laptop, no GPU needed.

In [ ]:
# Install libraries
%pip -q install sentence-transformers chromadb rank_bm25 pandas pyyaml

import csv, json, re, time
from collections import Counter
import numpy as np
import pandas as pd
import yaml

## 1. Load the settings and the chunks
All settings are in `config/rag.yaml` (course rule: parameters live there, not in the code).

In [ ]:
CONFIG = yaml.safe_load(open("../config/rag.yaml"))
EMB, STORE, RET, BM = CONFIG["embedding"], CONFIG["vector_store"], CONFIG["retrieval"], CONFIG["bm25"]

chunks = [json.loads(line) for line in open("../data/chunks.jsonl", encoding="utf-8")]
print(len(chunks), "chunks loaded")
{"embedding": EMB, "vector_store": STORE, "retrieval": RET, "bm25": BM}

## 2. Prepare the text
**What the embedding model reads:** `embed_text` = heading + the sentence that introduces the point + the full text of the provision (made in the chunking notebook). We do **not** remove stopwords here: the model was trained on normal sentences, and small words change the meaning of a law. For example *"shall **not** be considered to be high-risk"* means the opposite without "not".

**What keyword search (BM25) reads:** the same text in lower case, split into words, without stopwords. Stopwords such as "the", "of", "which" are in almost every chunk and do not help to tell chunks apart. We use the standard English list from scikit-learn, but **keep "not", "no" and "nor"**, because they change the meaning of a rule.

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

STOPWORDS = set(ENGLISH_STOP_WORDS) - set(BM["keep_words"])

def tokenize(text):
    """Lower case, words only (letters and numbers), without stopwords."""
    words = re.findall(r"[a-z0-9]+", text.lower())
    return [w for w in words if w not in STOPWORDS] if BM["stopwords"] else words

example = next(c for c in chunks if c["chunk_id"] == "ART6-3-1")
print("TEXT:  ", example["text"][:160], "...")
print("WORDS: ", tokenize(example["text"])[:20], "...")
print(len(STOPWORDS), "stopwords; kept:", BM["keep_words"])

**Comment:** _(write 1–2 sentences after running)_

## 3. Common words and phrases
Some words are in most chunks, for example "high-risk" or "referred to in". The instructor's advice: do **not** drop words just because they are frequent (for example with a TF-IDF cut-off), because a frequent word can still carry the meaning a question depends on. "High-risk" is in many chunks, but almost every question is about it.

So here we only **list** the candidates. Nothing is removed. Next step (planned): an LLM from a different model family than the answer model labels each candidate KEEP or IGNORE with a reason, and a team member makes the final decision.

In [ ]:
def share_of_chunks(items_per_chunk):
    """For each word or phrase: the share of chunks it appears in."""
    counts = Counter(item for items in items_per_chunk for item in set(items))
    return {item: n / len(chunks) for item, n in counts.items()}

words = [tokenize(c["text"]) for c in chunks]
phrases = [[" ".join(w[i:i + 2]) for i in range(len(w) - 1)] for w in words]   # two-word phrases

common_words = pd.Series(share_of_chunks(words)).sort_values(ascending=False).head(15)
common_phrases = pd.Series(share_of_chunks(phrases)).sort_values(ascending=False).head(15)
pd.DataFrame({"word": common_words.index, "share of chunks": common_words.round(2).values,
              "phrase": common_phrases.index, "share of chunks ": common_phrases.round(2).values})

**Comment:** _(write 1–2 sentences after running: which candidates look useless, which ones carry meaning?)_

## 4. Embed the chunks
`bge-base-en-v1.5` turns each text into 768 numbers. Texts with a similar meaning get similar numbers. We normalise the vectors, so cosine similarity is simply the dot product.

For **questions** the model card recommends a short instruction in front of the question (`query_instruction` in the config); the chunks are embedded without it.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(EMB["model"], device="cpu")

start = time.time()
vectors = model.encode([c["embed_text"] for c in chunks], batch_size=EMB["batch_size"],
                       normalize_embeddings=True, show_progress_bar=False)
print("vectors:", vectors.shape, f"in {time.time() - start:.1f} s")

def embed_question(question):
    """Vector of a question, with the instruction the model expects for search queries."""
    return model.encode([EMB["query_instruction"] + question], normalize_embeddings=True)[0]

**Quick check:** two points of the same rule should be closer to each other than to an unrelated provision.

In [ ]:
index_of = {c["chunk_id"]: i for i, c in enumerate(chunks)}
def similarity(a, b):
    return float(vectors[index_of[a]] @ vectors[index_of[b]])

pd.DataFrame([
    {"pair": "ART6-3-a vs ART6-3-b (two points of Article 6(3))", "cosine": similarity("ART6-3-a", "ART6-3-b")},
    {"pair": "ART6-3-a vs ART113-3-c-i (classification vs application date)", "cosine": similarity("ART6-3-a", "ART113-3-c-i")},
]).round(3)

**Comment:** _(write 1–2 sentences after running)_

## 5. Store the vectors in Chroma
Chroma keeps the vectors, the chunk texts and the metadata, and returns the closest chunks to a question vector. It is saved in `data/vector-store/` (not committed to Git; it is rebuilt by running this notebook).

In [ ]:
import chromadb

client = chromadb.PersistentClient(path=STORE["path"])
if STORE["collection"] in [c.name for c in client.list_collections()]:
    client.delete_collection(STORE["collection"])           # start clean on every run
collection = client.create_collection(STORE["collection"], metadata={"hnsw:space": STORE["distance"]})

META = ["ref", "title", "chapter", "area", "priority", "version_date", "prev_id", "next_id"]
collection.add(ids=[c["chunk_id"] for c in chunks],
               embeddings=vectors.tolist(),
               documents=[c["text"] for c in chunks],
               metadatas=[{k: str(c.get(k, "")) for k in META} for c in chunks])
print(collection.count(), "chunks in the vector store", STORE["path"])

## 6. Build the keyword index (BM25)
BM25 scores a chunk higher when it contains the question's words, especially rare ones. It finds exact terms such as "Annex III" or "Article 113" that the embedding model can miss.

In [ ]:
from rank_bm25 import BM25Okapi

bm25 = BM25Okapi([tokenize(c["embed_text"]) for c in chunks])
print("BM25 index over", len(chunks), "chunks")

## 7. Three ways to search
- **dense:** the closest vectors in Chroma (meaning)
- **bm25:** the best keyword scores (exact words)
- **hybrid:** both lists merged with Reciprocal Rank Fusion: each chunk gets `1 / (60 + rank)` from each list, and the scores are added. A chunk that is high in both lists comes first.

In [ ]:
def search_dense(question, n):
    """The n chunk IDs closest to the question vector."""
    result = collection.query(query_embeddings=[embed_question(question).tolist()], n_results=n)
    return result["ids"][0]

def search_bm25(question, n):
    """The n chunk IDs with the best BM25 score."""
    scores = bm25.get_scores(tokenize(question))
    return [chunks[i]["chunk_id"] for i in np.argsort(-scores)[:n]]

def search(question, mode=RET["mode"], k=RET["top_k"]):
    """The top-k chunk IDs for a question; mode is 'dense', 'bm25' or 'hybrid'."""
    if mode == "dense":
        return search_dense(question, k)
    if mode == "bm25":
        return search_bm25(question, k)
    rrf = Counter()
    for ranked in (search_dense(question, RET["candidates"]), search_bm25(question, RET["candidates"])):
        for rank, chunk_id in enumerate(ranked, start=1):
            rrf[chunk_id] += 1 / (RET["rrf_k"] + rank)
    return [chunk_id for chunk_id, _ in rrf.most_common(k)]

search("Is an AI system that grades students' exams high-risk?")

## 8. Try a handful of test questions
Six questions from `docs/eval/eval-set-v1.csv`. Next to the 5 retrieved chunks we show the **gold source** written by the team. A ✓ means a retrieved chunk is one of the gold provisions, by a simple text match of the references. This is only a first look, not the evaluation: Recall@k and MRR on the `val` questions are measured in KAN-99.

In [ ]:
questions = {r["id"]: r for r in csv.DictReader(open("../docs/eval/eval-set-v1.csv", encoding="utf-8-sig"))}
ref_of = {c["chunk_id"]: c["ref"] for c in chunks}

def matches_gold(ref, gold):
    """True if a chunk reference and a gold reference point to the same provision (one contains the other)."""
    for part in gold.split(";"):
        part = part.strip()
        if part.startswith(ref) or ref.startswith(part):
            return True
    return False

rows = []
for qid in ["BM-01", "BM-03", "BM-07", "B-03", "B-04", "EM-10"]:
    q = questions[qid]
    found = search(q["question"])
    rows.append({"id": qid, "question": q["question"][:90] + "...",
                 "top 5 (hybrid)": "\n".join(("✓ " if matches_gold(ref_of[i], q["gold_source"]) else "· ") + ref_of[i] for i in found),
                 "gold source": q["gold_source"]})
pd.set_option("display.max_colwidth", None)
pd.DataFrame(rows).style.set_properties(**{"white-space": "pre-wrap", "text-align": "left"})

**Comment:** _(write 1–2 sentences after running: for how many questions is a gold provision in the top 5?)_

## 9. Dense vs BM25 vs hybrid on one question
The same question with the three ways of searching. The design document plans this as two experiments: **E1** = dense only (baseline), **E2** = hybrid.

In [ ]:
q = questions["BM-07"]
print("QUESTION:", q["question"])
print("GOLD:    ", q["gold_source"], "\n")
pd.DataFrame({mode: [("✓ " if matches_gold(ref_of[i], q["gold_source"]) else "· ") + ref_of[i] for i in search(q["question"], mode)]
              for mode in ["dense", "bm25", "hybrid"]}, index=range(1, RET["top_k"] + 1))

**Comment:** _(write 1–2 sentences after running)_

## 10. What comes next
| Next step | Method from the instructor's slides | Jira |
|---|---|---|
| Measure retrieval on the `val` questions: **Recall@k** (did the gold provision come back in the top k?) and **MRR** (how high is the first correct chunk?) | *"most use an LLM judge, so also compute Recall@k / MRR on gold IDs"*; Context Recall → *"tune top-k and chunk size"*; Context Precision → *"are the useful chunks at the top of the list?"* | KAN-99 |
| E1 (dense) vs E2 (hybrid), logged in W&B with the settings | *"Log hyperparameters (prompt, chunk size, model) and compare runs"* | KAN-99 |
| Label the common words of step 3 with an LLM judge from a different model family, then a team member decides; test with and without them | instructor's feedback: no automatic TF-IDF cut | to be proposed |
| Answer metrics (Faithfulness, Context Entities Recall, Turn Contextual Relevancy) | RAGAS / DeepEval, need the generated answers | Sprint 3 |